In [ ]:
# Clean

from datetime import datetime
from getpass import getpass
import os

rdm_url = None
idp_name_1 = None
idp_username_1 = None
idp_password_1 = None
default_result_path = None
close_on_fail = False
transition_timeout = 30 * 1000
transition_timeout = 10 * 1000

project_name = None

binderhub_test_filename = 'grdm_test_file.txt'
delete_targets = [
    'postBuild',
    'apt.txt',
    'install.R',
    'Dockerfile',
    'environment.yml',
    'grdm_test_file.txt',
]


In [ ]:
if rdm_url is None:
    rdm_url = input(prompt=f'RDM URL: ')
if idp_name_1 is None:
    idp_name_1 = input(prompt=f'IdP Name: ')
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if project_name is None:
    project_name = datetime.now().strftime('TEST-BINDERHUB-%Y%m%d%H%M')

project_url = None
project_created = False

# BinderHubアドオン .bindeないのふぁいさくじょする

- サブシステム名: アドオン
- ページ/アドオン: BinderHub
- 機能分類: アドオン操作
- シナリオ名: repo2docker基本イメージを使った解析環境の起動
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM, BinderHub, GRDMは全てプロフィールを埋めていること / JupyterHubはサーバーが5つ以内の状態であること)、BinderHub OAuthクライアント情報
- 事前条件: 「プロジェクトに対するBinderHubアドオンの登録」を実施済みであること


In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url, wait_until="networkidle")
    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count():
        await consent_button.click()

await run_pw(_step)


## IdPを利用し、既存ユーザー1としてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)


## ダッシュボードから「新規プロジェクト作成」をクリックする

指定したプロジェクトが存在しない場合、新規プロジェクトが作成されること

In [ ]:
async def _step(page):
    global project_created
    project_created = await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)
    if project_created:
        print(f'Created project: {project_name}')
    else:
        print(f'Project already exists: {project_name}')

await run_pw(_step)


## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    global project_url
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    project_url = page.url

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「ファイル」をクリックする

フォルダ「.binder」が作成されていること

In [ ]:
async def _step(page):
    await page.locator('//li[@id = "projectNavFiles"]').click()
    binder_folder = page.locator('//*[text() = ".binder"]/../..//*[contains(@class, "fa-plus")]')
    await expect(binder_folder).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「.binder」フォルダを開く

`.binder`の頭につくアイコンが`-`になっていること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = ".binder"]/../..//*[contains(@class, "fa-plus")]').click()
    await expect(page.locator('//*[text() = ".binder"]/../..//*[contains(@class, "fa-minus")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「.binder」フォルダを開き、ファイルを削除する

apt.txt などのファイルを削除する

In [ ]:
import asyncio

async def _step(page):
    for filename in delete_targets:
        target = page.locator(f'//span[text() = "{filename}"]/../..')
        if await target.count() > 0:
            print(f'delete "{filename}"...')
            await expect(target).to_be_visible(timeout=transition_timeout)
            await target.click()
            delete_button = page.locator('//i[@class="fa fa-trash"]')
            await expect(delete_button).to_be_visible(timeout=transition_timeout)
            await delete_button.click()
            delete_confirm_button = page.locator('//span[@class="btn btn-danger"]')
            await expect(delete_confirm_button).to_be_visible(timeout=transition_timeout)
            await delete_confirm_button.click()
            await asyncio.sleep(3)

await run_pw(_step)


終了処理を実施する。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}